# 데코레이터와 컨텍스트 매니저

함수 호출 전후의 공통 처리와 자원 정리를 어떻게 안전하게 재사용하는지 확인합니다.

## 실행 환경 준비

이 노트북은 Python 3.12 표준 라이브러리만 사용합니다. A 경로 01~08절을 마친 뒤 선택하는 B 경로 확장 실습입니다.

In [1]:
# 필요한 표준 라이브러리를 가져옵니다.
import inspect
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from functools import wraps
from typing import Any, TypeVar

R = TypeVar("R")
print("Python 표준 라이브러리 준비 완료")

Python 표준 라이브러리 준비 완료


## 기본 실행: 함수의 앞뒤에 기록을 더하기

데코레이터는 함수를 받아 새 함수를 반환합니다. `wraps`는 원래 함수의 이름과 설명을 새 함수에 복사합니다.

In [2]:
# 원래 함수 정보를 유지하는 호출 기록 데코레이터를 만듭니다.
def trace(function: Callable[..., R]) -> Callable[..., R]:
    @wraps(function)
    def wrapper(*args: Any, **kwargs: Any) -> R:
        print(f"호출 시작: {function.__name__}")
        result = function(*args, **kwargs)
        print(f"호출 완료: {function.__name__}")
        return result
    return wrapper

@trace
def normalize_title(title: str) -> str:
    """제목 양쪽의 공백을 제거합니다."""
    return title.strip()

print("결과:", normalize_title("  월간 보고서  "))

호출 시작: normalize_title
호출 완료: normalize_title
결과: 월간 보고서


## 중간 결과 관찰: 원래 함수 정보 유지

디버거와 문서 생성기는 `__name__`, `__doc__`, 함수 서명을 읽습니다. `wraps`를 쓰면 장식된 함수에서도 이 정보를 확인할 수 있습니다.

In [3]:
# 장식된 함수의 이름·설명·서명을 확인합니다.
print("함수 이름:", normalize_title.__name__)
print("함수 설명:", normalize_title.__doc__)
print("함수 서명:", inspect.signature(normalize_title))
assert normalize_title.__name__ == "normalize_title"

함수 이름: normalize_title
함수 설명: 제목 양쪽의 공백을 제거합니다.
함수 서명: (title: str) -> str


## 값 하나를 바꿔 비교: 공백 없는 입력

같은 함수에 공백이 없는 제목을 전달합니다. 데코레이터가 입력값을 바꾸지 않고 원래 함수에 전달하는지 확인합니다.

In [4]:
# 공백이 없는 입력 한 가지만 바꿔 같은 함수를 실행합니다.
changed_title = normalize_title("월간 보고서")
print("공백 없는 결과:", changed_title)
assert changed_title == "월간 보고서"


호출 시작: normalize_title
호출 완료: normalize_title
공백 없는 결과: 월간 보고서


## Before: 예외가 발생하면 정리 코드에 도달하지 못함

`close()`를 정상 경로 끝에만 쓰면 작업 중 예외가 발생했을 때 자원이 열린 상태로 남습니다.

In [5]:
# 예외가 정리 문장보다 먼저 발생하는 상황을 재현합니다.
class DemoResource:
    def __init__(self) -> None:
        self.closed = False
    def close(self) -> None:
        self.closed = True

resource = DemoResource()
try:
    raise RuntimeError("처리 중 오류")
    resource.close()
except RuntimeError as error:
    print(type(error).__name__, "/ 닫힘:", resource.closed)
assert resource.closed is False

RuntimeError / 닫힘: False


## After: `try/finally`로 항상 정리하기

`contextmanager`로 만든 관리자는 `yield` 앞에서 자원을 준비하고 `finally`에서 정리합니다. `with` 블록에서 예외가 발생해도 `finally`가 실행됩니다.

In [6]:
# finally에서 자원을 닫는 컨텍스트 관리자를 만듭니다.
@contextmanager
def managed_resource() -> Iterator[DemoResource]:
    current = DemoResource()
    try:
        yield current
    finally:
        current.close()
        print("정리 실행")

managed: DemoResource | None = None
try:
    with managed_resource() as managed:
        raise RuntimeError("처리 중 오류")
except RuntimeError as error:
    print(type(error).__name__, "/ 닫힘:", managed.closed)
assert managed is not None and managed.closed is True

정리 실행
RuntimeError / 닫힘: True
